In [ ]:
# Sarah binti Shamsul Kamar (BIS01084600)

In [1]:
pip install autocorrect

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install textblob

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.0/625.0 kB 2.9 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install vaderSentiment

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install sklearn

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last reso

In [7]:
pip install tabulate

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import re
import string
import nltk
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from autocorrect import Speller
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tabulate import tabulate

# Download required NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt_tab')
nltk.download('punkt')

# ------TEXT PREPROCESSING-----
# Initialize tools
spell = Speller(lang='en')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Contractions dictionary
contractions_dict = {
    "wasn't": "was not", "isn't": "is not", "aren't": "are not",
    "weren't": "were not", "doesn't": "does not", "don't": "do not",
    "didn't": "did not", "can't": "cannot", "couldn't": "could not",
    "shouldn't": "should not", "wouldn't": "would not", "won't": "will not",
    "haven't": "have not", "hasn't": "has not", "hadn't": "had not",
    "i'm": "i am", "you're": "you are", "he's": "he is", "she's": "she is",
    "it's": "it is", "we're": "we are", "they're": "they are",
    "i've": "i have", "you've": "you have", "we've": "we have",
    "they've": "they have", "i'd": "i would", "you'd": "you would",
    "he'd": "he would", "she'd": "she would", "we'd": "we would",
    "they'd": "they would", "i'll": "i will", "you'll": "you will",
    "he'll": "he will", "she'll": "she will", "we'll": "we will",
    "they'll": "they will", "let's": "let us", "that's": "that is",
    "who's": "who is", "what's": "what is", "where's": "where is",
    "when's": "when is", "why's": "why is", "how's": "how is"
}

def remove_urls(text):
    """Remove URLs from text"""
    return re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

def remove_html(text):
    """Remove HTML tags from text"""
    try:
        return BeautifulSoup(text, "html.parser").get_text()
    except:
        return text

def remove_encoding_errors(text):
    """Remove encoding errors from text"""
    return re.sub(r'\b\w*[Ãâ¢€]\w*\b', '', text)

def replace_contractions(text):
    """Replace contractions with their expanded forms"""
    escaped_contractions = [re.escape(contraction) for contraction in contractions_dict.keys()]
    if escaped_contractions:
        contractions_pattern = r'\b(' + '|'.join(escaped_contractions) + r')\b'
        def replace_match(match):
            matched_word = match.group(0).lower()
            return contractions_dict.get(matched_word, matched_word)
        text = re.sub(contractions_pattern, replace_match, text, flags=re.IGNORECASE)
    return text

def clean_text(text):
    """Clean text by removing extra spaces and capitalizing sentences"""
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s+([,.!?])', r'\1', text)
    sentences = re.split(r'([.!?] )', text)
    for i in range(0, len(sentences), 2):
        if sentences[i]:
            sentences[i] = sentences[i].capitalize()
    text = ''.join(sentences)
    return text.strip()

def remove_punctuation_keep_spaces(text):
    """Remove punctuation but keep spaces"""
    return re.sub(r'[^\w\s]', ' ', text)

def remove_numbers(text):
    """Remove numbers from text"""
    return re.sub(r'\d+', '', text)

def correct_spelling(text):
    """Correct spelling errors"""
    try:
        words = text.split()
        corrected_words = []
        for word in words:
            if len(word) > 2 and word.isalpha():
                corrected = spell(word)
                corrected_words.append(corrected)
            else:
                corrected_words.append(word)
        return ' '.join(corrected_words)
    except:
        return text

def remove_stopwords(text):
    """Remove stopwords but keep important words for sentiment"""
    words = text.split()
    important_words = {'not', 'no', 'never', 'very', 'too', 'much', 'many', 'good', 'bad', 'great', 'terrible', 'amazing', 'horrible', 'excellent', 'poor', 'awesome', 'awful'}
    filtered_words = []
    for word in words:
        word_lower = word.lower()
        if word_lower not in stop_words or word_lower in important_words:
            filtered_words.append(word)
    return ' '.join(filtered_words)

def get_wordnet_pos(nltk_tag):
    """Map POS tags for lemmatization"""
    if nltk_tag.startswith('J'):
        return wordnet.ADJ
    elif nltk_tag.startswith('V'):
        return wordnet.VERB
    elif nltk_tag.startswith('N'):
        return wordnet.NOUN
    elif nltk_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def lemmatize_text(text):
    """Lemmatize text using POS tagging"""
    if not isinstance(text, str) or not text.strip():
        return ''
    try:
        words = word_tokenize(text)
        pos_tags = pos_tag(words)
        lemmatized_words = []
        for word, tag in pos_tags:
            if len(word) > 1:
                try:
                    lemmatized = lemmatizer.lemmatize(word, get_wordnet_pos(tag))
                    lemmatized_words.append(lemmatized)
                except:
                    lemmatized_words.append(word)
            else:
                lemmatized_words.append(word)
        return ' '.join(lemmatized_words)
    except:
        return text

def preprocess_text(text):
    """Main preprocessing function"""
    if pd.isna(text):
        return ""
    
    text = str(text)
    text = remove_urls(text)
    text = remove_html(text)
    text = remove_encoding_errors(text)
    text = replace_contractions(text)
    text = remove_numbers(text)
    text = remove_punctuation_keep_spaces(text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.lower()
    text = correct_spelling(text)
    text = remove_stopwords(text)
    text = lemmatize_text(text)
    text = clean_text(text)
    
    return text


# -------LOAD AND PREPROCESS DATA--------

# Load the dataset
try:
    df = pd.read_csv("Reviews.csv")
    print(f"\nDataset loaded successfully!")
    print(f"Original dataset shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
except FileNotFoundError:
    print("\nReviews.csv not found.")
    
    data_rows = []
    for i, (text, score) in enumerate(sample_reviews):
        data_rows.append({
            'Id': i,
            'ProductId': f'PROD{i}',
            'UserId': f'USER{i}',
            'ProfileName': f'User{i}',
            'HelpfulnessNumerator': 0,
            'HelpfulnessDenominator': 0,
            'Score': score,
            'Time': 1234567890,
            'Summary': text[:50],
            'Text': text
        })
    df = pd.DataFrame(data_rows)
    print(f"Created {len(df)} sample reviews")

# Sample 500 rows (or use all if less than 500)
sample_size = min(500, len(df))
sampled_df = df.sample(n=sample_size, random_state=42).copy()
print(f"\nSampled {sample_size} reviews for analysis")

# Convert Score to sentiment labels
# Score 1-2: negative (0), Score 3: neutral (1), Score 4-5: positive (2)
def score_to_sentiment(score):
    if score <= 2:
        return 0  # negative
    elif score == 3:
        return 1  # neutral
    else:
        return 2  # positive

def score_to_sentiment_label(score):
    if score <= 2:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

sampled_df['sentiment'] = sampled_df['Score'].apply(score_to_sentiment)
sampled_df['sentiment_label'] = sampled_df['Score'].apply(score_to_sentiment_label)

# Display sentiment distribution
print("\nSentiment Distribution in Sampled Data:")
sentiment_counts = sampled_df['sentiment_label'].value_counts()
for sentiment, count in sentiment_counts.items():
    print(f"  {sentiment}: {count} ({count/len(sampled_df)*100:.1f}%)")

# Apply preprocessing to Text column
print("\nPreprocessing text data...")
sampled_df['processed_text'] = sampled_df['Text'].apply(preprocess_text)

# Display sample of preprocessed text
print("\nSample of Original vs Processed Text:")
print("-" * 80)
sample_indices = sampled_df.sample(n=3, random_state=42).index
for idx in sample_indices:
    print(f"\nOriginal: {sampled_df.loc[idx, 'Text'][:100]}...")
    print(f"Processed: {sampled_df.loc[idx, 'processed_text'][:100]}...")
    print(f"Sentiment: {sampled_df.loc[idx, 'sentiment_label']} (Score: {sampled_df.loc[idx, 'Score']})")

# Save processed data
sampled_df.to_csv("Processed_Reviews.csv", index=False)
print("\nProcessed dataset saved as 'Processed_Reviews.csv'")


# -------LEXICON-BASED APPROACH (TextBlob & VADER)-------

# Initialize VADER analyzer
vader_analyzer = SentimentIntensityAnalyzer()

# Lists to store predictions
textblob_predictions = []
vader_predictions = []
actual_labels = []

print("\nAnalyzing sentiments using TextBlob and VADER...")

for idx, row in sampled_df.iterrows():
    text = row['processed_text']
    actual_label = row['sentiment_label']
    
    # TextBlob analysis
    try:
        blob = TextBlob(text)
        tb_polarity = blob.sentiment.polarity
        if tb_polarity > 0.05:
            tb_label = 'positive'
        elif tb_polarity < -0.05:
            tb_label = 'negative'
        else:
            tb_label = 'neutral'
    except:
        tb_label = 'neutral'
    
    # VADER analysis
    try:
        vader_scores = vader_analyzer.polarity_scores(text)
        vader_compound = vader_scores['compound']
        if vader_compound > 0.05:
            vader_label = 'positive'
        elif vader_compound < -0.05:
            vader_label = 'negative'
        else:
            vader_label = 'neutral'
    except:
        vader_label = 'neutral'
    
    textblob_predictions.append(tb_label)
    vader_predictions.append(vader_label)
    actual_labels.append(actual_label)

# Calculate classification reports
print("\n" + "-" * 40)
print("Classification Report - TextBlob:")
print("-" * 40)
tb_report = classification_report(actual_labels, textblob_predictions, 
                                   target_names=['negative', 'neutral', 'positive'])
print(tb_report)

print("-" * 40)
print("Classification Report - VADER:")
print("-" * 40)
vader_report = classification_report(actual_labels, vader_predictions,
                                      target_names=['negative', 'neutral', 'positive'])
print(vader_report)

# Calculate accuracy scores
tb_accuracy = accuracy_score(actual_labels, textblob_predictions)
vader_accuracy = accuracy_score(actual_labels, vader_predictions)

print("\nAccuracy Comparison:")
print(f"  TextBlob Accuracy: {tb_accuracy:.4f} ({tb_accuracy*100:.2f}%)")
print(f"  VADER Accuracy: {vader_accuracy:.4f} ({vader_accuracy*100:.2f}%)")


# -------FEATURE EXTRACTION (Bag-of-Words & TF-IDF)---------

# Prepare data for ML models
texts = sampled_df['processed_text'].tolist()
labels = sampled_df['sentiment_label'].tolist()

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"\nTraining set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

# Bag-of-Words Vectorization
print("\n1. Bag-of-Words Vectorization:")
bow_vectorizer = CountVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)
print(f"   Training features shape: {X_train_bow.shape}")
print(f"   Test features shape: {X_test_bow.shape}")
print(f"   Vocabulary size: {len(bow_vectorizer.vocabulary_)}")

# TF-IDF Vectorization
print("\n2. TF-IDF Vectorization:")
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)
print(f"   Training features shape: {X_train_tfidf.shape}")
print(f"   Test features shape: {X_test_tfidf.shape}")
print(f"   Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")

# Display top TF-IDF features
feature_names = tfidf_vectorizer.get_feature_names_out()
if len(feature_names) > 0:
    print("\n   Top 10 TF-IDF features:")
    # Get average TF-IDF scores across training data
    avg_tfidf = X_train_tfidf.mean(axis=0).A1
    top_indices = avg_tfidf.argsort()[-10:][::-1]
    for i, idx in enumerate(top_indices):
        if idx < len(feature_names):
            print(f"      {i+1}. {feature_names[idx]}: {avg_tfidf[idx]:.4f}")


# -------MACHINE LEARNING MODELS---------

# Dictionary to store results
results = {}

# Model 1: Naive Bayes with Bag-of-Words
print("\n1. Training Naive Bayes with Bag-of-Words...")
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)
y_pred_nb_bow = nb_bow.predict(X_test_bow)
results['Naive Bayes (BoW)'] = {
    'predictions': y_pred_nb_bow,
    'accuracy': accuracy_score(y_test, y_pred_nb_bow)
}

# Model 2: Naive Bayes with TF-IDF
print("2. Training Naive Bayes with TF-IDF...")
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)
y_pred_nb_tfidf = nb_tfidf.predict(X_test_tfidf)
results['Naive Bayes (TF-IDF)'] = {
    'predictions': y_pred_nb_tfidf,
    'accuracy': accuracy_score(y_test, y_pred_nb_tfidf)
}

# Model 3: SVM with Bag-of-Words
print("3. Training SVM with Bag-of-Words...")
svm_bow = SVC(kernel='linear', random_state=42)
svm_bow.fit(X_train_bow, y_train)
y_pred_svm_bow = svm_bow.predict(X_test_bow)
results['SVM (BoW)'] = {
    'predictions': y_pred_svm_bow,
    'accuracy': accuracy_score(y_test, y_pred_svm_bow)
}

# Model 4: SVM with TF-IDF
print("4. Training SVM with TF-IDF...")
svm_tfidf = SVC(kernel='linear', random_state=42)
svm_tfidf.fit(X_train_tfidf, y_train)
y_pred_svm_tfidf = svm_tfidf.predict(X_test_tfidf)
results['SVM (TF-IDF)'] = {
    'predictions': y_pred_svm_tfidf,
    'accuracy': accuracy_score(y_test, y_pred_svm_tfidf)
}

# Display accuracy comparison
print("\n" + "-" * 50)
print("MODEL ACCURACY COMPARISON:")
print("-" * 50)
for model_name, result in sorted(results.items(), key=lambda x: x[1]['accuracy'], reverse=True):
    print(f"  {model_name}: {result['accuracy']:.4f} ({result['accuracy']*100:.2f}%)")

# Detailed classification reports
for model_name, result in results.items():
    print("\n" + "-" * 40)
    print(f"Classification Report - {model_name}:")
    print("-" * 40)
    print(classification_report(y_test, result['predictions'], 
                                 target_names=['negative', 'neutral', 'positive']))
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, result['predictions'], 
                          labels=['negative', 'neutral', 'positive'])
    print(f"\nConfusion Matrix ({model_name}):")
    print("                 Predicted")
    print("              Neg  Neu  Pos")
    print(f"Actual Neg    {cm[0][0]:3d}  {cm[0][1]:3d}  {cm[0][2]:3d}")
    print(f"       Neu    {cm[1][0]:3d}  {cm[1][1]:3d}  {cm[1][2]:3d}")
    print(f"       Pos    {cm[2][0]:3d}  {cm[2][1]:3d}  {cm[2][2]:3d}")


# -----COMPREHENSIVE MODEL COMPARISON---------

# Create comparison table
comparison_data = []
for model_name, result in results.items():
    # Calculate error rate
    error_rate = 1 - result['accuracy']
    comparison_data.append([
        model_name,
        f"{result['accuracy']*100:.2f}%",
        f"{error_rate*100:.2f}%"
    ])

# Add lexicon-based models
comparison_data.append(["TextBlob (Lexicon)", f"{tb_accuracy*100:.2f}%", f"{(1-tb_accuracy)*100:.2f}%"])
comparison_data.append(["VADER (Lexicon)", f"{vader_accuracy*100:.2f}%", f"{(1-vader_accuracy)*100:.2f}%"])

print("\nModel Performance Summary:")
print(tabulate(comparison_data, 
               headers=["Model", "Accuracy", "Error Rate"], 
               tablefmt="grid"))

# Find best model
best_ml_model = max(results.items(), key=lambda x: x[1]['accuracy'])
print(f"\nBest Machine Learning Model: {best_ml_model[0]} with {best_ml_model[1]['accuracy']*100:.2f}% accuracy")

if tb_accuracy > best_ml_model[1]['accuracy']:
    print(f"Best Lexicon Model outperformed ML with {tb_accuracy*100:.2f}% accuracy")
else:
    print(f"Machine Learning outperforms Lexicon-based approaches by {(best_ml_model[1]['accuracy'] - tb_accuracy)*100:.2f}%")

# --------DISCUSSION-------

discussion = """
DISCUSSION: STRENGTHS AND WEAKNESSES OF SELECTED MODELS FOR SENTIMENT CLASSIFICATION

1. LEXICON-BASED APPROACHES (TextBlob & VADER)

STRENGTHS:
- No training data required
- High processing efficiency
- Works well for general sentiment without domain adaptation
- VADER is specifically tuned for social media large datasets
- Easy to understand why a document is positive or negative based on word scores

WEAKNESSES:
- Cannot capture context-dependent sentiment (e.g., "This is sick!" - could be positive or negative)
- Struggles with sarcasm and irony
- Difficulty with neutral sentiment
- Limited vocabulary
- Lower accuracy compared to ML approaches

2. MACHINE LEARNING APPROACHES (Naive Bayes & SVM)

Naive Bayes Strengths:
- Fast training and prediction
- High positive sentiment detection
- Easy to implement
- Ideal for real-time applications and large-scale datasets

Naive Bayes Weaknesses:
- Poor negative sentiment recall
- Assumes feature dependence
- Severely biased towards majority class
- Cannot capture word dependencies

SVM Strengths:
- Higher overall accuracy
- Better class balance
- Handles high dimensional data
- Can capture complex word relationships
- Better negative sentiment detection

SVM Weaknesses:
- Requires longer processing data time, especially on large datasets
- Requires careful parameter tuning
- Less interpreable than Naive Bayes

CONCLUSION:
For this Amazon reviews dataset, machine learning approaches (particularly SVM with TF-IDF) 
demonstrate superior performance compared to lexicon-based methods. 
"""

print(discussion)

# Save discussion to file
with open("discussion.txt", "w") as f:
    f.write(discussion)
print("\nDiscussion saved to 'discussion.txt'")


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/4af25032-c966-47e7-803b-
[nltk_data]     397ac7e4a5e7/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/4af25032-c966-47e7-803b-
[nltk_data]     397ac7e4a5e7/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/4af25032-c966-47e7-803b-
[nltk_data]     397ac7e4a5e7/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/4af25032-c966-47e7-803b-
[nltk_data]     397ac7e4a5e7/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/4af25032-c966-47e7-803b-
[nltk_data]     397ac7e4a5e7/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[n


Dataset loaded successfully!
Original dataset shape: (568454, 10)
Columns: ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']

Sampled 500 reviews for analysis

Sentiment Distribution in Sampled Data:
  positive: 397 (79.4%)
  negative: 74 (14.8%)
  neutral: 29 (5.8%)

Preprocessing text data...

Sample of Original vs Processed Text:
--------------------------------------------------------------------------------

Original: Simply excellent! Great tasting and works. I have been using this for months with success. I highly ...
Processed: Simply excellent great taste work use month success highly recommend product...
Sentiment: positive (Score: 5)

Original: We use chicken bouillon in a lot of our home cooked meals.<br />Finding organic bouillon is oft time...
Processed: Use chicken bouillon lot home cook meal find organic bouillon oft time challenge pleased find produc...
Sentiment: positive (Score: 5)

Ori

/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier,